# Joint Inference Without NumPyro

This notebook shows the new standalone path-scoring workflow for direct joint state and parameter inference.

The central primitive is `dsx.path_log_prob(...)`, which returns the full path score

$$\log p(x_0) + \sum_k \log p(x_{k+1} \mid x_k) + \sum_k \log p(y_k \mid x_k).$$

Because it does not enter a NumPyro trace, it works naturally with `jax.grad`, `optax`, custom optimizers, and other standalone workflows.

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import numpyro.distributions as dist

import dynestyx as dsx


## A small discrete-time model

We'll use a simple latent AR(1)-style model with Gaussian transitions and Gaussian observations.

In [2]:
def make_dynamics(alpha):
    return dsx.DynamicalModel(
        control_dim=0,
        initial_condition=dist.MultivariateNormal(jnp.zeros(1), 0.25 * jnp.eye(1)),
        state_evolution=dsx.GaussianStateEvolution(
            F=lambda x, u, t_now, t_next: alpha * x,
            cov=jnp.array([[0.05]]),
        ),
        observation_model=dsx.LinearGaussianObservation(
            H=jnp.array([[1.0]]),
            R=jnp.array([[0.08]]),
        ),
    )


times = jnp.arange(20.0)
true_alpha = 0.72
sim = dsx.simulate(make_dynamics(true_alpha), predict_times=times, rng_key=jr.PRNGKey(0))
true_states = sim.states[0]
obs_values = sim.observations[0]


## Standalone path objective

For direct optimization, we treat both the latent path and the parameter as variables.

`chunk_size` lets us evaluate the observation and transition terms as a scan of vmaps instead of one large vmap. For short trajectories you can leave it as `None`; for long trajectories it helps keep memory bounded.

In [3]:
def neg_log_joint(alpha, latent_path):
    dynamics = make_dynamics(alpha)
    return -dsx.path_log_prob(
        dynamics,
        latent_path,
        obs_times=times,
        obs_values=obs_values,
        chunk_size=8,
    )


## Optimizing states and parameters together

This is the basic pattern:

1. choose an initial latent path and parameter value,
2. differentiate `neg_log_joint`, and
3. update both with your optimizer of choice.

In [4]:
alpha = jnp.array(0.3)
latent_path = jnp.zeros_like(true_states)
optimizer = optax.adam(1e-2)
opt_state = optimizer.init((alpha, latent_path))

grad_fn = jax.grad(neg_log_joint, argnums=(0, 1))

for _ in range(50):
    grads = grad_fn(alpha, latent_path)
    updates, opt_state = optimizer.update(grads, opt_state, (alpha, latent_path))
    alpha, latent_path = optax.apply_updates((alpha, latent_path), updates)

alpha, latent_path[:3]


(Array(0.6976493, dtype=float32),
 Array([[0.39564455],
        [0.25920272],
        [0.09990835]], dtype=float32))

## When to prefer this workflow

This standalone path-scoring route is most attractive when:

- you want to use optimization/MCMC methods that are not native to Numpyro/BlackJAX,
- you want to differentiate through the full latent path objective,
- or you want custom control over chunking / batching outside a NumPyro trace.

For structure-exploiting marginal inference, `Filter` and `Smoother` are still the preferred tools.